# Parametric Studies - Demo_IEEE14 Package Sweep

This notebook sweeps one parameter of a package model, simulating the package at each value and comparing a chosen output variable across the sweep.

Set `REINITIALIZE_EACH_CASE` to choose whether the package is initialized once before the sweep, or reinitialized after each parameter update.

In [ ]:
import logging
import os
import shutil
import sys

sys.path.append(os.path.abspath(".."))

logging.getLogger("OMPython").setLevel(logging.ERROR)

import pandas as pd
from OMPython import ModelicaSystemOMC, OMCSessionLocal

from scripts.helpers import *
from scripts.parametric_study_helpers import case_label, load_modelica_file
from scripts.package_model_initialization import initialize_loaded_package_model

## User Configuration

Set the sweep component, parameter, values, and the model paths here.

In [ ]:
MODEL = "Demo_IEEE14.IEEE14DisconnectLine"
SOURCE_PACKAGE = MODEL.split(".")[0]
MODEL_DIR = os.path.abspath(os.path.join("..", "FullWorkflow", "models", SOURCE_PACKAGE))
MODELS_PKG_PATH = os.path.join(MODEL_DIR, "package.mo")

OMLIB_DIR = os.path.join(os.path.expanduser("~"), ".openmodelica", "libraries")
os.environ["OPENMODELICALIBRARY"] = OMLIB_DIR
MODELICA_PKG_PATH = os.path.join(OMLIB_DIR, "Modelica 3.2.3+maint.om", "package.mo")
DYNAWO_PKG_PATH = os.path.abspath(
    os.path.join("..", "..", "julia_openmodelica", "dynawo_library", "Dynawo", "package.mo")
)

SWEEP_COMPONENT = "LineB2B5"
SWEEP_PARAMETER = "XPu"
SWEEP_VALUES = [0.12, 0.173884267, 0.24]

REINITIALIZE_EACH_CASE = True

INIT_MODEL_BY_COMPONENT = {}
SLACK_COMPONENT = "Gen1"

PLOT_VARIABLE = "LineB2B5.P1Pu"
OUTPUT_DIR = os.path.abspath(os.path.join("outputs", "Demo_IEEE14_package_parametric_runs"))

## Prepare OpenModelica Session

The output directories are created and the library paths are checked before the sweep runs.

In [ ]:
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "modelica")
RESULT_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "results")

if not os.path.isfile(MODELS_PKG_PATH):
    raise RuntimeError(f"Package file not found: {MODELS_PKG_PATH}")
if not os.path.isfile(DYNAWO_PKG_PATH):
    raise RuntimeError(f"Dynawo package not found: {DYNAWO_PKG_PATH}")
if not os.path.isfile(MODELICA_PKG_PATH):
    raise RuntimeError(f"Modelica package not found: {MODELICA_PKG_PATH}")

shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULT_OUTPUT_DIR, exist_ok=True)

StudyOMC = OMCSessionLocal()
load_modelica_file(
    StudyOMC,
    MODELS_PKG_PATH,
    MODELICA_PKG_PATH,
    DYNAWO_PKG_PATH,
)
config = check_user_configuration_package(
    StudyOMC,
    model=MODEL,
    sweep_component=SWEEP_COMPONENT,
    sweep_parameter=SWEEP_PARAMETER,
    slack_component=SLACK_COMPONENT,
    init_model_by_component=INIT_MODEL_BY_COMPONENT,
)
MODEL_CHAIN = config["model_chain"]
PARAMETER_MODEL = config["parameter_model"]

## Initialize the Base Case

When `REINITIALIZE_EACH_CASE` is `False`, the package is initialized once here and reused for every sweep value.

In [ ]:
if not REINITIALIZE_EACH_CASE:
    base_initialized_case = initialize_loaded_package_model(
        StudyOMC,
        source_model=MODEL,
        model_chain=MODEL_CHAIN,
        case_name=SOURCE_PACKAGE + "_base",
        output_dir=OUTPUT_DIR,
        modelica_package_path=MODELICA_PKG_PATH,
        dynawo_package_path=DYNAWO_PKG_PATH,
        init_model_by_component=INIT_MODEL_BY_COMPONENT,
        slack_component=SLACK_COMPONENT,
    )

    base_initialized_package_file = base_initialized_case["initialized_package_file"]
    base_initialized_model = base_initialized_case["initialized_model"]

    base_initialized_model_map = initialized_name_map(
        MODEL_CHAIN,
        base_initialized_case["initialized_package"],
    )
    base_parameter_model = base_initialized_model_map[PARAMETER_MODEL]

StudyOMC.sendExpression("quit()")

## Run the Parameter Sweep

Each sweep value is applied, the package is simulated, and the selected variable is collected for comparison.

In [ ]:
sweep_results = []

for value in SWEEP_VALUES:
    label = case_label(value)
    parameter_name = SWEEP_PARAMETER.replace(".", "_")
    case_name = "_".join([SOURCE_PACKAGE, SWEEP_COMPONENT, parameter_name, label])
    selected_resultfile = os.path.join(RESULT_OUTPUT_DIR, case_name + "_selected.csv")

    StudyOMC = OMCSessionLocal()

    if REINITIALIZE_EACH_CASE:
        load_modelica_file(
            StudyOMC,
            MODELS_PKG_PATH,
            MODELICA_PKG_PATH,
            DYNAWO_PKG_PATH,
        )
        omc_call(
            StudyOMC,
            f"setParameterValue({PARAMETER_MODEL}, {SWEEP_COMPONENT}.{SWEEP_PARAMETER}, {value})",
            parsed=False,
        )

        initialized_case = initialize_loaded_package_model(
            StudyOMC,
            source_model=MODEL,
            model_chain=MODEL_CHAIN,
            case_name=case_name,
            output_dir=OUTPUT_DIR,
            modelica_package_path=MODELICA_PKG_PATH,
            dynawo_package_path=DYNAWO_PKG_PATH,
            init_model_by_component=INIT_MODEL_BY_COMPONENT,
            slack_component=SLACK_COMPONENT,
        )

        simulation_model = initialized_case["initialized_model"]
    else:
        load_modelica_file(
            StudyOMC,
            base_initialized_package_file,
            MODELICA_PKG_PATH,
            DYNAWO_PKG_PATH,
        )
        omc_call(
            StudyOMC,
            f"setParameterValue({base_parameter_model}, {SWEEP_COMPONENT}.{SWEEP_PARAMETER}, {value})",
            parsed=False,
        )

        simulation_model = base_initialized_model

    result_prefix = case_name + "_initialized"

    StudySystem = ModelicaSystemOMC(session=StudyOMC)
    StudySystem.model(model_name=simulation_model)

    simflags = simulation_flags_without_log_stats(StudyOMC, simulation_model)
    omc_call(
        StudyOMC,
        f'simulate({simulation_model}, outputFormat="csv", '
        f'fileNamePrefix="{result_prefix}", simflags="{simflags}")',
        parsed=False,
    )

    raw_resultfile = os.path.join(str(StudySystem.getWorkDirectory()), result_prefix + "_res.csv")
    if PLOT_VARIABLE not in pd.read_csv(raw_resultfile, nrows=0).columns:
        raise RuntimeError(f'PLOT_VARIABLE "{PLOT_VARIABLE}" is not a variable in the simulation result.')
    df = pd.read_csv(raw_resultfile, usecols=["time", PLOT_VARIABLE])
    df["case"] = label
    df["parameter_value"] = value

    df.to_csv(selected_resultfile, index=False)
    sweep_results.append(df)
    StudyOMC.sendExpression("quit()")

    print(f"Finished case {label} with {SWEEP_COMPONENT}.{SWEEP_PARAMETER} = {value}")

sweep_df = pd.concat(sweep_results, ignore_index=True)

print("Stored selected result files in:", RESULT_OUTPUT_DIR)

## Compare Sweep Results

The selected variable is plotted across all sweep values.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
ax.set_title(f"Package sweep: {SWEEP_COMPONENT}.{SWEEP_PARAMETER}")
ax.set_xlabel("Time (s)")
ax.set_ylabel(PLOT_VARIABLE)

for label, case_group in sweep_df.groupby("case"):
    value = case_group["parameter_value"].iloc[0]
    ax.plot(case_group["time"], case_group[PLOT_VARIABLE], label=f"{SWEEP_PARAMETER} = {value}")

ax.legend(loc="lower right")